In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import Tool
from langchain.chains import LLMMathChain
from langchain_community.tools import DuckDuckGoSearchRun

# 加载环境变量
load_dotenv()

# 初始化 LLM (大脑)
llm = ChatOpenAI(temperature=0, model="gpt-4o")

# 初始化工具 (手和脚)

# 工具1：搜索
# DuckDuckGoSearchRun 是一个封装好的工具，可以直接使用
search = DuckDuckGoSearchRun()
search_tool = Tool(
    name="Search",
    func=search.run,
    description="一个搜索引擎。当你需要回答关于时事、事实或你不知道的信息时，应该使用它。输入应该是具体的问题或搜索词。"
)

# 工具2：计算器
# LLMMathChain 可以让 LLM 来执行数学运算
llm_math_chain = LLMMathChain.from_llm(llm=llm, verbose=False)
calculator_tool = Tool(
    name="Calculator",
    func=llm_math_chain.run,
    description="一个计算器。当你需要进行数学计算或回答关于数学的问题时使用。输入应该是一个完整的数学表达式。"
)

# 将所有工具放入一个列表中
tools = [search_tool, calculator_tool]

print("LLM 和工具准备完毕！")
print(f"共配置了 {len(tools)} 个工具。")

LLM 和工具准备完毕！
共配置了 2 个工具。


In [7]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# 1. 从 LangChain Hub 中拉取预设的 ReAct 提示词模板
# 这是构建 Agent 的标准做法，确保了提示词的质量和兼容性
prompt = hub.pull("hwchase17/react")

# 2. 创建 Agent (大脑的逻辑)
#    这个函数将 LLM、工具列表和提示词组合在一起，创建出 Agent 的核心逻辑
agent = create_react_agent(llm, tools, prompt)

# 3. 创建 Agent 执行器 (驱动循环的身体)
#    - agent: 上一步创建的 Agent 逻辑
#    - tools: Agent 可以使用的工具列表
#    - verbose=True: 这是最重要的参数！它会打印出 Agent 的完整思考过程，便于我们观察和理解
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agent 和执行器创建成功！")

/opt/anaconda3/envs/langchain-mcp/lib/python3.11/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Agent 和执行器创建成功！


In [8]:
# 提出一个需要结合搜索和计算才能回答的问题
# 为了让结果稳定，我们明确指定今天的日期
input_question = "日本天皇现在几岁？请将今天的日期按照2025年6月10日计算。"

# 使用 .invoke() 方法运行 AgentExecutor
result = agent_executor.invoke({"input": input_question})

print("\n\n----------- 最终答案 -----------")
print(result["output"])



> Entering new AgentExecutor chain...
要计算日本天皇的年龄，我们需要知道他的出生日期以及计算的日期。日本现任天皇是德仁天皇，他出生于1960年2月23日。我们需要计算从1960年2月23日到2025年6月10日的年龄。

Action: Calculator
Action Input: 2025 - 1960Answer: 65我们已经计算出从1960年到2025年间的年份差为65年。接下来，我们需要考虑具体的月份和日期，以确定德仁天皇在2025年6月10日的确切年龄。

德仁天皇出生于1960年2月23日，因此在2025年2月23日他将满65岁。从2月23日到6月10日，他已经过了65岁生日。

Final Answer: 到2025年6月10日，日本天皇德仁将满65岁。

> Finished chain.


----------- 最终答案 -----------
到2025年6月10日，日本天皇德仁将满65岁。
